# 04 - Data split

Builds the **one** train/test split and the cross-validation folds that `05_Model` uses.

**How:** rows are split at random, **stratified by curing age**, so every age (3, 7, 28, 90, ... days) has the same share in train, test and every CV fold. Strength depends strongly on age, so this keeps all splits comparable.

**What the test score then means:** most mixes were tested at several ages, so a mix can be in training at one age and in the test set at another. The test score measures how well the model predicts **a known mix at a new age** (the usual setup in published work on this dataset). How well it does on completely new mix designs is reported separately in `05_Model` (section 7) for context.

Output: `data/splits.csv` (inside `ML/`): the raw columns plus `mix_id`, `split` and `fold`. Engineered features are **not** stored here; they are computed by `ML/features.py` inside the model pipeline, so training and the app use identical code.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.model_selection import StratifiedKFold, train_test_split

sys.path.insert(0, str(Path('..').resolve()))   # backend/ -> "ML.features" is importable
from ML.features import INGREDIENTS, RAW_FEATURES, TARGET

SEED = 42
DATA_PATH = Path('..') / 'Data' / 'concrete_data.csv'   # read only
OUT_DIR = Path('data')
OUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.width', 200)

## 1. Load

Columns are renamed to the names the app uses (`slag` → `blast_furnace_slag`, `strength` → `compressive_strength`), and the 25 exact duplicate rows are dropped before anything is split, so an identical row can't land in both train and test.

In [2]:
df = pd.read_csv(DATA_PATH).rename(columns={'slag': 'blast_furnace_slag', 'strength': 'compressive_strength'})
n_raw = len(df)
df = df.drop_duplicates().reset_index(drop=True)
assert list(df.columns) == RAW_FEATURES + [TARGET]
print(f'{n_raw} rows, {n_raw - len(df)} duplicates dropped, {len(df)} remain')

1030 rows, 25 duplicates dropped, 1005 remain


## 2. Mix id (for reference)

Rows with identical values for all 7 ingredients are the same mix design. The id is **not** used for splitting; it is kept so `05_Model` can report the new-mix score and run per-mix sanity checks.

In [3]:
df['mix_id'] = df.groupby(INGREDIENTS).ngroup()
print('Distinct mix designs:', df['mix_id'].nunique())
print('Ages tested per mix :', df.groupby('mix_id').size().value_counts().sort_index().to_dict())

Distinct mix designs: 428
Ages tested per mix : {1: 246, 2: 42, 3: 7, 4: 38, 5: 77, 6: 12, 7: 3, 8: 3}


## 3. Age strata

One stratum per curing age. Ages with fewer than 10 rows (1, 120 and 360 days) are too rare to spread over 5 folds, so each is merged with its nearest common age (1 → 3, 120 → 100, 360 → 365).

In [4]:
age_counts = df['age'].value_counts()
common = np.sort(age_counts[age_counts >= 10].index.to_numpy())
nearest = {a: int(common[np.abs(common - a).argmin()]) for a in age_counts.index}
strata = df['age'].map(nearest)

print('Merged ages:', {a: n for a, n in nearest.items() if a != n})
pd.DataFrame({'rows': strata.value_counts().sort_index()}).T

Merged ages: {360: 365, 120: 100, 1: 3}


age,3,7,14,28,56,90,91,100,180,270,365
rows,131,122,62,419,86,54,17,55,26,13,20


## 4. Hold-out test set (20% of rows)

Stratified by age. It is used **once**, at the very end of `05_Model`.

In [5]:
train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.2, stratify=strata, random_state=SEED)
df['split'] = 'train'
df.loc[test_idx, 'split'] = 'test'
print(df['split'].value_counts().to_dict())

seen = df.loc[test_idx, 'mix_id'].isin(set(df.loc[train_idx, 'mix_id']))
print(f'Test rows whose mix also appears in train at another age: {seen.sum()} of {len(test_idx)} ({seen.mean() * 100:.0f}%)')

{'train': 804, 'test': 201}
Test rows whose mix also appears in train at another age: 148 of 201 (74%)


In [6]:
print('Rows per age (%):')
display((pd.crosstab(df['age'], df['split'], normalize='columns') * 100).round(1).T)

tr, te = df[df['split'] == 'train'], df[df['split'] == 'test']
balance = pd.DataFrame({'train_mean': tr[RAW_FEATURES + [TARGET]].mean(), 'test_mean': te[RAW_FEATURES + [TARGET]].mean(),
                        'ks_pvalue': [ks_2samp(tr[c], te[c]).pvalue for c in RAW_FEATURES + [TARGET]]})
balance['similar (p>0.05)'] = balance['ks_pvalue'] > 0.05
balance.round(3)

Rows per age (%):


age,1,3,7,14,28,56,90,91,100,120,180,270,360,365
split,,,,,,,,,,,,,,
test,0.5,12.4,12.4,6.0,41.8,8.5,5.5,1.5,5.5,0.0,2.5,1.5,1.0,1.0
train,0.1,12.9,12.1,6.2,41.7,8.6,5.3,1.7,5.1,0.4,2.6,1.2,0.5,1.5


,train_mean,test_mean,ks_pvalue,similar (p>0.05)
cement,281.426,267.440,0.207,True
blast_furnace_slag,69.912,80.567,0.475,True
fly_ash,54.840,58.314,0.221,True
water,181.929,182.655,0.913,True
superplasticizer,6.061,5.913,0.342,True
coarse_aggregate,974.469,974.005,0.656,True
fine_aggregate,773.334,770.096,0.475,True
age,45.821,46.000,1.000,True
compressive_strength,35.484,34.317,0.895,True


## 5. Cross-validation folds (inside the training set)

Five folds over the training rows, again stratified by age. All model comparison and tuning in `05_Model` uses exactly these folds, so every model is scored on the same validation rows.

In [7]:
train_pos = np.flatnonzero(df['split'].eq('train').to_numpy())
df['fold'] = 'test'
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for k, (_, val) in enumerate(skf.split(train_pos, strata.iloc[train_pos]), start=1):
    df.loc[train_pos[val], 'fold'] = f'fold_{k}'
assert (df.loc[train_pos, 'fold'] != 'test').all()

age_group = pd.cut(df['age'], bins=[0, 7, 28, 100, 365], labels=['1-7', '14-28', '56-100', '120-365'])
summary = df.groupby('fold').agg(rows=(TARGET, 'size'), mixes=('mix_id', 'nunique'),
                                 strength_mean=(TARGET, 'mean'), strength_std=(TARGET, 'std'))
summary.join((pd.crosstab(df['fold'], age_group, normalize='index') * 100).round(1)).round(2)

,rows,mixes,strength_mean,strength_std,1-7,14-28,56-100,120-365
fold,,,,,,,,
fold_1,161,137,35.42,15.78,25.5,47.8,20.5,6.2
fold_2,161,133,37.17,16.60,25.5,47.8,21.1,5.6
fold_3,161,129,34.37,16.76,24.8,47.8,19.9,7.5
fold_4,161,140,35.22,16.31,24.8,47.8,21.7,5.6
fold_5,160,131,35.23,16.72,25.0,48.1,20.6,6.2
test,201,152,34.32,15.72,25.4,47.8,20.9,6.0


## 6. Save

In [8]:
out = df[RAW_FEATURES + [TARGET, 'mix_id', 'split', 'fold']]
out_path = OUT_DIR / 'splits.csv'
out.to_csv(out_path, index=False)

check = pd.read_csv(out_path)
assert check.shape == out.shape
print(f'Saved {out_path}  {check.shape}')
check['fold'].value_counts().sort_index()

Saved data\splits.csv  (1005, 12)


fold
fold_1    161
fold_2    161
fold_3    161
fold_4    161
fold_5    160
test      201
Name: count, dtype: int64